In [1]:
import confnotebook

In [2]:
from pathlib import Path

source = Path("../examples/RPA-6542/")

files = sorted(source.glob("*.pdf"))

for i, file in enumerate(files):
    print(f"[{i}] {file.stem}")

[0] 18470938
[1] 18470982
[2] 18548791
[3] 18561292
[4] 18561867
[5] 18639563
[6] 18660877
[7] 18667876
[8] 18667914
[9] 18668755
[10] 18674893
[11] 18687424
[12] 18690095
[13] 18690959
[14] 18692621
[15] 18699963
[16] [Untitled]_23-48


In [3]:
IDX_FILE = 6

In [4]:
from vision_core.debug_image_observer import DebugImageObserver

file = files[IDX_FILE]
output_dir = f"../examples/output/{file.stem}"

debug_image_observer = DebugImageObserver(output_dir=output_dir)

/mnt/data/projects/rusal_recon_srv/repo/recon_vision/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `DISABLE_MODEL_SOURCE_CHECK` to `True`.


In [5]:
from vision_core.pipelines.build_document import DocumentBuildPipeline

pipeline = DocumentBuildPipeline(debug_image=debug_image_observer)

document = pipeline.build(file.read_bytes())

/mnt/data/projects/rusal_recon_srv/repo/recon_vision/.venv/lib/python3.11/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv5_server_det', '/mnt/data/projects/rusal_recon_srv/repo/recon_vision/models/PP-OCRv5_server_det')
Creating model: ('cyrillic_PP-OCRv5_mobile_rec', '/mnt/data/projects/rusal_recon_srv/repo/recon_vision/models/cyrillic_PP-OCRv5_mobile_rec')
2026-07-10 11:09:42.239 | INFO     | vision_core.pipelines.build_document:build:104 - Обработка страницы 0 с dpi 200...
2026-07-10 11:09:42.386 | INFO     | vision_core.pipelines.build_document:_process_page:186 - Коррекция ориентации и наклона...
2026-07-10 11:09:42.870 | DEBUG    | vision_core.preprocessor.image_orientation:process:47 - Ориентация страницы: 0° 

In [6]:
from app.infrastructure.services.structured_data_extractor import ReconciliationActExtractor

extractor = ReconciliationActExtractor()
data = await extractor.extract(document)

2026-07-10 11:09:57.512 | DEBUG    | app.infrastructure.services.extractor.company_ext:_build_summary_text:70 - summary_text: 1316 символов из 5 страниц
2026-07-10 11:09:57.512 | DEBUG    | vision_core.postprocessor.dc_cols_resolver:_is_similar_keyword:52 - Проверяем похожесть 'ао "русал братск"' и 'дебет' (ratio=0.12)
2026-07-10 11:09:57.513 | DEBUG    | vision_core.postprocessor.dc_cols_resolver:_is_similar_keyword:52 - Проверяем похожесть 'ао "русал братск"' и 'кредит' (ratio=0.12)
2026-07-10 11:09:57.513 | DEBUG    | vision_core.postprocessor.dc_cols_resolver:_is_similar_keyword:52 - Проверяем похожесть 'ао "русал вами" (0000007120)' и 'дебет' (ratio=0.00)
2026-07-10 11:09:57.513 | DEBUG    | vision_core.postprocessor.dc_cols_resolver:_is_similar_keyword:52 - Проверяем похожесть 'ао "русал вами" (0000007120)' и 'кредит' (ratio=0.07)
2026-07-10 11:09:57.513 | DEBUG    | app.infrastructure.services.extractor.company_ext:_build_summary_cell_texts:93 - summary_cell_texts: ['АО "РУСАЛ Б

In [7]:
print(data.debit)

[LedgerEntry(record='САЛЬДО НА 01.08.2023, В ВАЛЮТЕ РУБ.', value=0.0, date='01.08.2023', row_reference=RowReference(id_table='0', id_row='2', id_col=3, buyer_col=5)), LedgerEntry(record='П/П № 42136, РАЗРАБОТКА РД ОБЪЕКТОВ RUB', value=15573819.54, date='16.11.2022', row_reference=RowReference(id_table='0', id_row='3', id_col=3, buyer_col=5)), LedgerEntry(record='С/Ф № 00000000625, ТЕХНИЧЕСКОЕ СОПРОВОЖДЕНИЕ РД С АВГУСТА ПО ОКТЯБРЬ RUB', value=0.0, date='15.11.2023', row_reference=RowReference(id_table='0', id_row='4', id_col=3, buyer_col=5)), LedgerEntry(record='П/П № 42302, ОПЛАТА ПО ДОГ. 019-23-ПА ОТ 02.10.2023 ПО СЧ/Ф 00000000625 ОТ 15.11.2023 ТЕХСОПРОВОЖДЕНИЕ РД ПЕРИОД 01.08.2023-30.10.2023, В ТОМ ЧИСЛЕ НДС 20% RUB', value=4641160.83, date='07.12.2023', row_reference=RowReference(id_table='0', id_row='5', id_col=3, buyer_col=5)), LedgerEntry(record='С/Ф № 00000001075, ТЕХНИЧ. СОПРОВОЖД. РД С ЯНВАРЯ ПО ИЮЛЬ RUB', value=0.0, date='27.12.2023', row_reference=RowReference(id_table='0', 

In [8]:
from app.application.dto.fill_reconciliation_act import FillReconciliationActCommand
from app.domain.entities.process import ProcessState
from app.infrastructure.services.pdf_filler import DocumentPdfFiller

process_state = ProcessState(
    process_id="notebook-test",
    source_pdf=files[IDX_FILE].read_bytes(),
    document_payload=document,
)

comments = f"""
            По данным покупателя {data.buyer}
            По данным продавца {data.seller}
            В период: {data.period.start} - {data.period.end}
            """

# используем значения продавца для заполнения колонок покупателя
command = FillReconciliationActCommand(
    process_id="notebook-test",
    comments=comments,
    debit=data.debit,
    credit=data.credit,
)

filler = DocumentPdfFiller()
filled_pdf = await filler.fill(process_state, command)

2026-07-10 11:10:09.570 | INFO     | app.infrastructure.services.pdf_fill.render:resolve_font_file:221 - найден шрифт: /mnt/data/projects/rusal_recon_srv/repo/recon_vision/assets/fonts/LiberationSerif-Regular.ttf
2026-07-10 11:10:09.691 | DEBUG    | app.infrastructure.services.pdf_fill.render:load_aligned_page_images:49 - page=0 dpi=200 source=1654x2339 aligned=1654x2339 canvas=1654x2339
2026-07-10 11:10:09.819 | DEBUG    | app.infrastructure.services.pdf_fill.render:load_aligned_page_images:49 - page=1 dpi=200 source=1654x2339 aligned=1654x2339 canvas=1654x2339
2026-07-10 11:10:09.950 | DEBUG    | app.infrastructure.services.pdf_fill.render:load_aligned_page_images:49 - page=2 dpi=200 source=1654x2339 aligned=1654x2339 canvas=1654x2339
2026-07-10 11:10:10.077 | DEBUG    | app.infrastructure.services.pdf_fill.render:load_aligned_page_images:49 - page=3 dpi=200 source=1654x2339 aligned=1654x2339 canvas=1654x2339
2026-07-10 11:10:10.193 | DEBUG    | app.infrastructure.services.pdf_fill.r

In [9]:
out_path = f"../examples/output/{files[IDX_FILE].stem}_filled.pdf"
Path(out_path).write_bytes(filled_pdf)
print(out_path)

../examples/output/18660877_filled.pdf
